# Step 8: Evaluation + Reasons
Read the top flags; plant **seen** and **unseen** anomalies and report detection rate separately (MF3); compare with the rules baseline at the same number of flags (MF4); flagged real rows are "unverified"; compute reasons and finalize EN/AR texts.

**Input:** `data/processed/features.csv + models/isolation_forest.joblib`  
**Output:** `evaluation results, final reason texts`

In [1]:
# 8.1 Load the saved model and its settings from Step 7, and the reason texts (EN + AR) from src/predict.py,
# the same texts the ML service returns, so there's one copy of them.
# model_input() and apply_rules() are copied from notebook 06 for now; in Part B they move to src/features.py.
import json
import sys
import joblib
import numpy as np
import pandas as pd

sys.path.insert(0, "..")            # lets the notebook import from the ml/src folder
from src.predict import REASONS     # code -> (English text, Arabic text)

df = pd.read_csv("../data/processed/features.csv", parse_dates=["expense_date"])
model = joblib.load("../models/isolation_forest.joblib")
with open("../models/model_config.json", encoding="utf-8") as fh:
    cfg = json.load(fh)
MODEL_FEATURES = cfg["model_features"]
print(cfg["model_version"], "|", len(df), "requests |", len(REASONS), "reason codes")

def model_input(data):
    # One-sided amounts (Step 7, 7.8), using the floors saved in the config
    X = data[MODEL_FEATURES].copy()
    X["log_amount"] = X["log_amount"].clip(lower=cfg["model_input"]["log_amount_floor"])
    X["amount_vs_employee_category"] = X["amount_vs_employee_category"].clip(lower=0)
    return X

def apply_rules(data):
    # The 3 rules (Step 7, 7.13), using the settings and limits saved in the config
    r, limits = cfg["rules"], cfg["category_limits"]
    same_day = data.groupby(["requester_id", "expense_date", "category"])["amount_sar"]
    total, smallest, largest = same_day.transform("sum"), same_day.transform("min"), same_day.transform("max")
    limit = data["category"].map(limits)
    n = data["same_category_day_count"]
    split = ((n >= 2) & (n <= r["split_max_parts"]) & (largest <= limit) & (total > limit)
             & (smallest >= r["split_min_part"] * limit))
    duplicate = (data["duplicate_count"].between(2, r["duplicate_max_copies"])
                 & (data["amount_sar"] >= r["duplicate_min_amount"]))
    many = data["other_category_day_count"] >= r["many_other_from"]
    return pd.DataFrame({"possible_split_request": split, "duplicate_request": duplicate,
                         "many_requests_same_day": many}, index=data.index)

iforest-0.1 | 209012 requests | 10 reason codes


_Code for this step comes next._

In [2]:
# 8.2 Same scoring as notebook 06 (7.15), from the saved files only.
X = model_input(df)
df["score"] = -model.score_samples(X)
rules = apply_rules(df)
cut = cfg["cut_offs"]
df["risk_level"] = np.where(df["score"] >= cut["high_from"], "high",
                   np.where((df["score"] >= cut["medium_from"]) | rules.any(axis=1), "medium", "low"))
print(df["risk_level"].value_counts())

risk_level
low       200492
medium      6428
high        2092
Name: count, dtype: int64


In [3]:
# 8.3 For every flagged request and every model feature: set that feature to a NORMAL value,
# score again, and measure how much the score dropped. A big drop = that feature is a reason.
# NORMAL values = what a typical request has: the amount floor, 0 spreads above normal,
# the median category share, not weekend, not round.
flagged = df["risk_level"] != "low"
X_flag = X[flagged]
score_flag = df.loc[flagged, "score"]

NORMAL = {
    "log_amount": cfg["model_input"]["log_amount_floor"],
    "amount_vs_employee_category": 0.0,
    "category_share_for_employee": float(X["category_share_for_employee"].median()),
    "is_weekend": 0,
    "is_round_amount": 0,
}
drop = pd.DataFrame(index=X_flag.index)
for feature, normal_value in NORMAL.items():
    what_if = X_flag.copy()
    what_if[feature] = normal_value                       # only this one feature changes
    drop[feature] = score_flag - (-model.score_samples(what_if))
print(drop.describe().T[["mean", "50%", "max"]].round(3))

                              mean    50%    max
log_amount                   0.020  0.008  0.260
amount_vs_employee_category  0.013  0.000  0.291
category_share_for_employee  0.015  0.011  0.087
is_weekend                   0.036  0.000  0.173
is_round_amount              0.019  0.000  0.203


In [4]:
# 8.4 Reasons for each flagged request:
#  1. Every rule that fired is a reason (strength 1.0, so rules come first: they're plain facts).
#  2. A model feature is a reason when its "what if" drop is at least MIN_DROP AND it makes sense:
#     the amount is really above normal, the category is really rare for THIS person (they have
#     their own history), the day really is a weekend, the amount really is round.
#     The amount reason's code depends on what it was compared with (baseline_level, Step 6).
#  3. Strongest first, at most 3 (the contract's limit). If nothing passes MIN_DROP, the single
#     strongest sensible feature is used, so every flagged request has at least one reason.
MIN_DROP = 0.02
MAX_REASONS = 3
UNUSUAL_SHARE = 0.2          # "rare category" = under 20% of this person's earlier requests
AMOUNT_CODE = {"empcat": "amount_high_for_employee_category",
               "emp": "amount_high_for_employee",
               "deptcat": "amount_high_for_department"}

def reasons_for(row, drops, fired_rules):
    candidates = {code: 1.0 for code, fired in fired_rules.items() if fired}
    checks = {   # reason code: (feature it comes from, does this reason make sense here?)
        "large_amount": ("log_amount", row["log_amount"] > cfg["model_input"]["log_amount_floor"]),
        AMOUNT_CODE.get(row["baseline_level"], "none"): ("amount_vs_employee_category",
            row["amount_vs_employee_category"] >= 1 and row["baseline_level"] in AMOUNT_CODE),
        "unusual_category_for_employee": ("category_share_for_employee",
            row["category_share_for_employee"] <= UNUSUAL_SHARE and row["baseline_level"] in ("empcat", "emp")),
        "weekend_expense": ("is_weekend", row["is_weekend"] == 1),
        "round_amount": ("is_round_amount", row["is_round_amount"] == 1),
    }
    model_candidates = {code: drops[feature] for code, (feature, makes_sense) in checks.items() if makes_sense}
    candidates.update({code: d for code, d in model_candidates.items() if d >= MIN_DROP})
    if not candidates and model_candidates:          # fallback: the strongest sensible one
        best = max(model_candidates, key=model_candidates.get)
        candidates[best] = model_candidates[best]
    return sorted(candidates, key=candidates.get, reverse=True)[:MAX_REASONS]

flag_rows = df[flagged]
codes = pd.Series([reasons_for(row, drop.loc[i], rules.loc[i]) for i, row in flag_rows.iterrows()],
                  index=flag_rows.index)
df["reason_codes"] = pd.Series([[] for _ in range(len(df))], index=df.index)   # low risk = no reasons
df.loc[flagged, "reason_codes"] = codes

print("Flagged:", flagged.sum(), "| without a reason:", (codes.str.len() == 0).sum())
print("Reasons per flagged request:", codes.str.len().value_counts().sort_index().to_dict())
print(codes.explode().value_counts())

Flagged: 8520 | without a reason: 0
Reasons per flagged request: {1: 3175, 2: 4235, 3: 1110}
large_amount                         3103
duplicate_request                    2569
weekend_expense                      2481
amount_high_for_employee_category    1582
possible_split_request               1415
unusual_category_for_employee        1252
round_amount                         1102
many_requests_same_day                802
amount_high_for_employee              530
amount_high_for_department            139
Name: count, dtype: int64


In [5]:
# 8.5 The exact shape the ML service returns and the backend saves in risk_justification:
# [{"code", "text_en", "text_ar"}, ...]
def reason_details(code_list):
    return [{"code": c, "text_en": REASONS[c][0], "text_ar": REASONS[c][1]} for c in code_list]

df["reason_details"] = df["reason_codes"].apply(reason_details)

ex = df.sort_values("score", ascending=False).iloc[2]          # the 3rd most unusual request
print(ex["requester_id"], ex["category"], ex["amount_sar"], ex["risk_level"])
for d in ex["reason_details"]:
    print(" -", d["code"], "|", d["text_en"], "|", d["text_ar"])

3525 Travel 1125.0 high
 - amount_high_for_employee_category | Amount is much higher than this employee usually requests in this category | المبلغ أعلى بكثير مما يطلبه هذا الموظف عادةً في هذه الفئة
 - round_amount | Round amount (e.g. 500 or 1,000) | المبلغ رقم مقرّب (مثل 500 أو 1,000)
 - weekend_expense | Expense date falls on a weekend (Friday–Saturday) | تاريخ المصروف في عطلة نهاية الأسبوع (الجمعة–السبت)


In [6]:
# 8.6 The 6 most unusual requests and 5 random medium ones, with their reasons.
cols = ["requester_id", "category", "amount_sar", "risk_level", "reason_codes"]
print(df.sort_values("score", ascending=False)[cols].head(6).to_string(index=False))
print(df[df["risk_level"] == "medium"].sample(5, random_state=1)[cols].to_string(index=False))

 requester_id                category  amount_sar risk_level                                                       reason_codes
         3440 Equipment & Maintenance    183750.0       high    [round_amount, large_amount, amount_high_for_employee_category]
         1197            IT & Telecom     18750.0       high             [round_amount, large_amount, amount_high_for_employee]
         3525                  Travel      1125.0       high [amount_high_for_employee_category, round_amount, weekend_expense]
         4020                  Travel      5625.0       high [round_amount, amount_high_for_employee_category, weekend_expense]
         2878            IT & Telecom      9375.0       high          [round_amount, amount_high_for_employee, weekend_expense]
         5235            IT & Telecom     25125.0       high                      [round_amount, weekend_expense, large_amount]
 requester_id category  amount_sar risk_level                                                   reason_c